In [0]:
# ============================================
# SILVER_AGG – Station Hour Flow Aggregation
# User Story: US10 / Feature Engineering Base
# ============================================

from pyspark.sql import functions as F

# --------------------------------------------
# 1) Paths configuration
# --------------------------------------------

SILVER_TRIPS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/bikeshare_trips"
SILVER_AGG_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/station_hour_flow"

# --------------------------------------------
# 2) Read Silver trip-level dataset
# --------------------------------------------

df = spark.read.parquet(SILVER_TRIPS)
print("Input rows:", df.count())

# --------------------------------------------
# 3) Helper: convert "HH:00" → integer hour
# --------------------------------------------

def hour_str_to_int(col_name: str):
    """
    Extract hour from string format HH:00
    Example: '13:00' → 13
    """
    return F.regexp_extract(F.col(col_name), r"^(\d{1,2})", 1).cast("int")

# --------------------------------------------
# 4) Departures aggregation
# --------------------------------------------

dep = (
    df
    .withColumn("hour", hour_str_to_int("hour_start_str"))
    .groupBy(
        F.col("start_station_id").alias("station_id"),
        "year",
        "month",
        "day",
        F.col("hour_start_str").alias("hour_str"),
        "hour"
    )
    .agg(F.count("*").alias("departures"))
)
print("Departures rows:", dep.count())

# --------------------------------------------
# 5) Arrivals aggregation
# --------------------------------------------

arr = (
    df
    .withColumn("hour", hour_str_to_int("hour_end_str"))
    .groupBy(
        F.col("end_station_id").alias("station_id"),
        "year",
        "month",
        "day",
        F.col("hour_end_str").alias("hour_str"),
        "hour"
    )
    .agg(F.count("*").alias("arrivals"))
)
print("Arrivals rows:", arr.count())

# --------------------------------------------
# 6) Combine departures + arrivals
# --------------------------------------------

station_hour = (
    dep.join(
        arr,
        ["station_id", "year", "month", "day", "hour_str", "hour"],
        "full"
    )
    .na.fill(0, ["departures", "arrivals"])
    .withColumn("net_flow", F.col("arrivals") - F.col("departures"))
)

# --------------------------------------------
# 7) Data sanity filters
# --------------------------------------------

station_hour = station_hour.filter(
    (F.col("hour").between(0, 23)) &
    (F.col("day").between(1, 31))
)

# --------------------------------------------
# 8) Data Quality Validation
# --------------------------------------------

dq_nulls = station_hour.select(
    F.count(F.when(F.col("station_id").isNull(), True)).alias("null_station_id"),
    F.count(F.when(F.col("year").isNull(), True)).alias("null_year"),
    F.count(F.when(F.col("month").isNull(), True)).alias("null_month"),
    F.count(F.when(F.col("day").isNull(), True)).alias("null_day"),
    F.count(F.when(F.col("hour").isNull(), True)).alias("null_hour")
)
display(dq_nulls)

# --------------------------------------------
# 9) Remove null key records (production ready)
# --------------------------------------------

station_hour = station_hour.filter(
    F.col("station_id").isNotNull() &
    F.col("year").isNotNull() &
    F.col("month").isNotNull() &
    F.col("day").isNotNull() &
    F.col("hour").isNotNull()
)
print("Rows after DQ filter:", station_hour.count())

# --------------------------------------------
# 10) Write partitioned dataset
# --------------------------------------------

(
    station_hour
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(SILVER_AGG_DIR)
)
print("Written to:", SILVER_AGG_DIR)

# --------------------------------------------
# 11) Validation evidence
# --------------------------------------------

# Sample records
display(
    station_hour
    .orderBy("year", "month", "day", "hour")
    .limit(10)
)

# Records per partition
display(
    station_hour
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month")
)

# Net flow distribution
display(
    station_hour
    .select("net_flow")
    .summary("min", "25%", "50%", "75%", "max")
)


Input rows: 12028835
Departures rows: 4426675
Arrivals rows: 4370779


null_station_id,null_year,null_month,null_day,null_hour
0,0,0,0,0


Rows after DQ filter: 5725483
Written to: dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver_agg/station_hour_flow


station_id,year,month,day,hour_str,hour,departures,arrivals,net_flow
7321,2022,10,1,00:00,0,1,2,1
7023,2022,10,1,00:00,0,1,1,0
7284,2022,10,1,00:00,0,3,0,-3
7245,2022,10,1,00:00,0,2,2,0
7238,2022,10,1,00:00,0,3,4,1
7237,2022,10,1,00:00,0,1,0,-1
7102,2022,10,1,00:00,0,8,0,-8
7044,2022,10,1,00:00,0,2,1,-1
7698,2022,10,1,00:00,0,1,0,-1
7286,2022,10,1,00:00,0,1,0,-1


year,month,count
2022,10,243023
2022,11,195579
2022,12,146779
2023,1,148717
2023,2,133332
2023,3,163224
2023,4,214058
2023,5,261184
2023,6,270813
2023,7,288624


summary,net_flow
min,-171
25%,-1
50%,0
75%,1
max,175
